In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
)

print("Non-IID experiment notebook ready")

Non-IID experiment notebook ready


In [2]:
SEEDS = [11, 22, 33, 44, 55]

N_CLIENTS = 3
N_ROUNDS = 5
LOCAL_EPOCHS = 5
LEARNING_RATE = 0.01
CLASSES = np.array([0, 1])

print("Non-IID experiment settings loaded")
print("Seeds:", SEEDS)
print("Clients:", N_CLIENTS)
print("Rounds:", N_ROUNDS)
print("Local epochs:", LOCAL_EPOCHS)

Non-IID experiment settings loaded
Seeds: [11, 22, 33, 44, 55]
Clients: 3
Rounds: 5
Local epochs: 5


In [3]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
    }


def initialise_sgd_model(n_features, seed):
    model = SGDClassifier(
        loss="log_loss",
        learning_rate="constant",
        eta0=LEARNING_RATE,
        alpha=0.0001,
        max_iter=1,
        tol=None,
        random_state=seed,
        fit_intercept=True,
    )

    model.partial_fit(
        np.zeros((1, n_features)),
        np.array([0]),
        classes=CLASSES
    )
    return model


def get_sgd_parameters(model):
    return [model.coef_.copy(), model.intercept_.copy()]


def set_sgd_parameters(model, parameters):
    model.coef_ = parameters[0].copy()
    model.intercept_ = parameters[1].copy()
    model.classes_ = CLASSES
    return model


def weighted_average_parameters(client_parameters, client_sizes):
    total_size = sum(client_sizes)

    avg_coef = sum(
        params[0] * size for params, size in zip(client_parameters, client_sizes)
    ) / total_size

    avg_intercept = sum(
        params[1] * size for params, size in zip(client_parameters, client_sizes)
    ) / total_size

    return [avg_coef, avg_intercept]


print("Helper functions loaded")

Helper functions loaded


In [4]:
def load_heart_dataset():
    !pip -q install ucimlrepo
    from ucimlrepo import fetch_ucirepo

    heart = fetch_ucirepo(id=45)

    X = heart.data.features
    y_raw = heart.data.targets

    y = (y_raw.iloc[:, 0] > 0).astype(int)

    data = X.copy()
    data["target"] = y
    data = data.dropna()

    X = data.drop(columns=["target"]).astype(float)
    y = data["target"].astype(int)

    return X, y


def load_stroke_dataset():
    !pip -q install kagglehub
    import kagglehub
    import glob
    import os

    dataset_path = kagglehub.dataset_download(
        "fedesoriano/stroke-prediction-dataset"
    )

    stroke_file = glob.glob(
        os.path.join(dataset_path, "healthcare-dataset-stroke-data.csv")
    )[0]

    stroke_data = pd.read_csv(stroke_file)

    if "id" in stroke_data.columns:
        stroke_data = stroke_data.drop(columns=["id"])

    stroke_data = stroke_data.dropna().copy()

    y = stroke_data["stroke"].astype(int)
    X = stroke_data.drop(columns=["stroke"])

    X = pd.get_dummies(X, drop_first=True)
    X = X.astype(float)

    return X, y


print("Dataset-loading functions ready")

Dataset-loading functions ready


In [5]:
def create_non_iid_hospital_split(X_train_scaled, y_train, seed):
    """
    Create 3 simulated hospitals with different class distributions.

    Hospital A: higher proportion of positive cases
    Hospital B: lower proportion of positive cases
    Hospital C: mixed distribution
    """

    rng = np.random.default_rng(seed)

    y_array = y_train.to_numpy()

    positive_indices = np.where(y_array == 1)[0]
    negative_indices = np.where(y_array == 0)[0]

    rng.shuffle(positive_indices)
    rng.shuffle(negative_indices)

    # Split positive cases unevenly: 50%, 20%, 30%
    pos_a, pos_b, pos_c = np.split(
        positive_indices,
        [
            int(0.50 * len(positive_indices)),
            int(0.70 * len(positive_indices)),
        ]
    )

    # Split negative cases unevenly: 25%, 45%, 30%
    neg_a, neg_b, neg_c = np.split(
        negative_indices,
        [
            int(0.25 * len(negative_indices)),
            int(0.70 * len(negative_indices)),
        ]
    )

    hosp_a_indices = np.concatenate([pos_a, neg_a])
    hosp_b_indices = np.concatenate([pos_b, neg_b])
    hosp_c_indices = np.concatenate([pos_c, neg_c])

    rng.shuffle(hosp_a_indices)
    rng.shuffle(hosp_b_indices)
    rng.shuffle(hosp_c_indices)

    hospital_data = {
        "Hospital_A_high_positive": (
            X_train_scaled[hosp_a_indices],
            y_array[hosp_a_indices],
        ),
        "Hospital_B_low_positive": (
            X_train_scaled[hosp_b_indices],
            y_array[hosp_b_indices],
        ),
        "Hospital_C_mixed": (
            X_train_scaled[hosp_c_indices],
            y_array[hosp_c_indices],
        ),
    }

    return hospital_data


print("Non-IID hospital split function ready")

Non-IID hospital split function ready


In [7]:
def run_non_iid_experiment(dataset_name, X, y, use_class_weight=False):
    all_results = []
    partition_logs = []

    for seed in SEEDS:
        print(f"Running non-IID {dataset_name} experiment with seed {seed}")

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.20,
            random_state=seed,
            stratify=y
        )

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        class_weight = "balanced" if use_class_weight else None

        central_model = LogisticRegression(
            max_iter=2000,
            random_state=seed,
            class_weight=class_weight,
            solver="liblinear"
        )

        start = time.time()
        central_model.fit(X_train_scaled, y_train)
        central_time = time.time() - start

        central_pred = central_model.predict(X_test_scaled)
        central_prob = central_model.predict_proba(X_test_scaled)[:, 1]

        central_metrics = compute_metrics(y_test, central_pred, central_prob)
        central_metrics.update({
            "Dataset": dataset_name,
            "Seed": seed,
            "Split": "Non-IID",
            "Method": "Centralised LR",
            "Training Time (s)": central_time,
        })

        all_results.append(central_metrics)

        hospital_data = create_non_iid_hospital_split(
            X_train_scaled,
            y_train,
            seed
        )

        for hospital_name, (_, y_local) in hospital_data.items():
            partition_logs.append({
                "Dataset": dataset_name,
                "Seed": seed,
                "Hospital": hospital_name,
                "Samples": len(y_local),
                "Negative Cases": int(np.sum(y_local == 0)),
                "Positive Cases": int(np.sum(y_local == 1)),
                "Positive Rate": float(np.mean(y_local)),
            })

        if use_class_weight:
            negative_count = np.sum(y_train.to_numpy() == 0)
            positive_count = np.sum(y_train.to_numpy() == 1)
            positive_weight = negative_count / positive_count
        else:
            positive_weight = 1.0

        def train_local_model(X_local, y_local, global_parameters):
            local_model = initialise_sgd_model(X_local.shape[1], seed)
            local_model = set_sgd_parameters(local_model, global_parameters)

            sample_weights = np.where(y_local == 1, positive_weight, 1.0)

            for _ in range(LOCAL_EPOCHS):
                local_model.partial_fit(
                    X_local,
                    y_local,
                    classes=CLASSES,
                    sample_weight=sample_weights
                )

            return local_model

        fed_start = time.time()

        global_model = initialise_sgd_model(X_train_scaled.shape[1], seed)
        global_parameters = get_sgd_parameters(global_model)

        for _ in range(N_ROUNDS):
            client_parameters = []
            client_sizes = []

            for _, (X_local, y_local) in hospital_data.items():
                local_model = train_local_model(
                    X_local,
                    y_local,
                    global_parameters
                )

                client_parameters.append(get_sgd_parameters(local_model))
                client_sizes.append(len(y_local))

            global_parameters = weighted_average_parameters(
                client_parameters,
                client_sizes
            )

            global_model = set_sgd_parameters(global_model, global_parameters)

        fed_time = time.time() - fed_start

        fed_pred = global_model.predict(X_test_scaled)
        fed_prob = global_model.predict_proba(X_test_scaled)[:, 1]

        fed_metrics = compute_metrics(y_test, fed_pred, fed_prob)
        fed_metrics.update({
            "Dataset": dataset_name,
            "Seed": seed,
            "Split": "Non-IID",
            "Method": "Federated LR (FedAvg)",
            "Training Time (s)": fed_time,
        })

        all_results.append(fed_metrics)

    return pd.DataFrame(all_results), pd.DataFrame(partition_logs)


print("Non-IID experiment runner ready")

Non-IID experiment runner ready


In [8]:
heart_X, heart_y = load_heart_dataset()

heart_non_iid_results, heart_partition_logs = run_non_iid_experiment(
    dataset_name="Heart Disease",
    X=heart_X,
    y=heart_y,
    use_class_weight=False
)

print("Heart Disease non-IID results")
display(heart_non_iid_results.round(4))

print("Heart Disease hospital partition logs")
display(heart_partition_logs.round(4))

Running non-IID Heart Disease experiment with seed 11
Running non-IID Heart Disease experiment with seed 22
Running non-IID Heart Disease experiment with seed 33
Running non-IID Heart Disease experiment with seed 44
Running non-IID Heart Disease experiment with seed 55
Heart Disease non-IID results


,Accuracy,Precision,Recall,F1-score,Balanced Accuracy,ROC-AUC,Dataset,Seed,Split,Method,Training Time (s)
0,0.8333,0.8750,0.7500,0.8077,0.8281,0.8560,Heart Disease,11,Non-IID,Centralised LR,0.0025
1,0.8167,0.8696,0.7143,0.7843,0.8103,0.8493,Heart Disease,11,Non-IID,Federated LR (FedAvg),0.0883
2,0.8500,0.8519,0.8214,0.8364,0.8482,0.9118,Heart Disease,22,Non-IID,Centralised LR,0.0017
3,0.8500,0.8519,0.8214,0.8364,0.8482,0.9208,Heart Disease,22,Non-IID,Federated LR (FedAvg),0.0929
4,0.8500,0.8519,0.8214,0.8364,0.8482,0.9241,Heart Disease,33,Non-IID,Centralised LR,0.0017
5,0.8667,0.8571,0.8571,0.8571,0.8661,0.9263,Heart Disease,33,Non-IID,Federated LR (FedAvg),0.0880
6,0.7667,0.7917,0.6786,0.7308,0.7612,0.8426,Heart Disease,44,Non-IID,Centralised LR,0.0017
7,0.7833,0.8261,0.6786,0.7451,0.7768,0.8426,Heart Disease,44,Non-IID,Federated LR (FedAvg),0.0940
8,0.9000,0.8929,0.8929,0.8929,0.8996,0.9554,Heart Disease,55,Non-IID,Centralised LR,0.0016
9,0.9167,0.9259,0.8929,0.9091,0.9152,0.9565,Heart Disease,55,Non-IID,Federated LR (FedAvg),0.0868


Heart Disease hospital partition logs


,Dataset,Seed,Hospital,Samples,Negative Cases,Positive Cases,Positive Rate
0,Heart Disease,11,Hospital_A_high_positive,86,32,54,0.6279
1,Heart Disease,11,Hospital_B_low_positive,79,57,22,0.2785
2,Heart Disease,11,Hospital_C_mixed,72,39,33,0.4583
3,Heart Disease,22,Hospital_A_high_positive,86,32,54,0.6279
4,Heart Disease,22,Hospital_B_low_positive,79,57,22,0.2785
5,Heart Disease,22,Hospital_C_mixed,72,39,33,0.4583
6,Heart Disease,33,Hospital_A_high_positive,86,32,54,0.6279
7,Heart Disease,33,Hospital_B_low_positive,79,57,22,0.2785
8,Heart Disease,33,Hospital_C_mixed,72,39,33,0.4583
9,Heart Disease,44,Hospital_A_high_positive,86,32,54,0.6279


In [9]:
stroke_X, stroke_y = load_stroke_dataset()

stroke_non_iid_results, stroke_partition_logs = run_non_iid_experiment(
    dataset_name="Stroke Prediction",
    X=stroke_X,
    y=stroke_y,
    use_class_weight=True
)

print("Stroke Prediction non-IID results")
display(stroke_non_iid_results.round(4))

print("Stroke hospital partition logs")
display(stroke_partition_logs.round(4))

Using Colab cache for faster access to the 'stroke-prediction-dataset' dataset.
Running non-IID Stroke Prediction experiment with seed 11
Running non-IID Stroke Prediction experiment with seed 22
Running non-IID Stroke Prediction experiment with seed 33
Running non-IID Stroke Prediction experiment with seed 44
Running non-IID Stroke Prediction experiment with seed 55
Stroke Prediction non-IID results


,Accuracy,Precision,Recall,F1-score,Balanced Accuracy,ROC-AUC,Dataset,Seed,Split,Method,Training Time (s)
0,0.7434,0.1168,0.7619,0.2025,0.7522,0.8217,Stroke Prediction,11,Non-IID,Centralised LR,0.0199
1,0.7780,0.1174,0.6429,0.1985,0.7134,0.7995,Stroke Prediction,11,Non-IID,Federated LR (FedAvg),0.1383
2,0.7617,0.1279,0.7857,0.2200,0.7732,0.8422,Stroke Prediction,22,Non-IID,Centralised LR,0.0236
3,0.7923,0.1318,0.6905,0.2214,0.7436,0.8256,Stroke Prediction,22,Non-IID,Federated LR (FedAvg),0.1182
4,0.7363,0.1322,0.9286,0.2315,0.8281,0.8752,Stroke Prediction,33,Non-IID,Centralised LR,0.0169
5,0.7678,0.1423,0.8810,0.2450,0.8219,0.8787,Stroke Prediction,33,Non-IID,Federated LR (FedAvg),0.1109
6,0.7495,0.1222,0.7857,0.2115,0.7668,0.8582,Stroke Prediction,44,Non-IID,Centralised LR,0.0188
7,0.8014,0.1408,0.7143,0.2353,0.7598,0.8532,Stroke Prediction,44,Non-IID,Federated LR (FedAvg),0.1124
8,0.7576,0.1260,0.7857,0.2171,0.7710,0.8254,Stroke Prediction,55,Non-IID,Centralised LR,0.0157
9,0.7729,0.1276,0.7381,0.2175,0.7563,0.8270,Stroke Prediction,55,Non-IID,Federated LR (FedAvg),0.1299


Stroke hospital partition logs


,Dataset,Seed,Hospital,Samples,Negative Cases,Positive Cases,Positive Rate
0,Stroke Prediction,11,Hospital_A_high_positive,1023,940,83,0.0811
1,Stroke Prediction,11,Hospital_B_low_positive,1725,1692,33,0.0191
2,Stroke Prediction,11,Hospital_C_mixed,1179,1128,51,0.0433
3,Stroke Prediction,22,Hospital_A_high_positive,1023,940,83,0.0811
4,Stroke Prediction,22,Hospital_B_low_positive,1725,1692,33,0.0191
5,Stroke Prediction,22,Hospital_C_mixed,1179,1128,51,0.0433
6,Stroke Prediction,33,Hospital_A_high_positive,1023,940,83,0.0811
7,Stroke Prediction,33,Hospital_B_low_positive,1725,1692,33,0.0191
8,Stroke Prediction,33,Hospital_C_mixed,1179,1128,51,0.0433
9,Stroke Prediction,44,Hospital_A_high_positive,1023,940,83,0.0811


In [10]:
all_non_iid_results = pd.concat(
    [heart_non_iid_results, stroke_non_iid_results],
    ignore_index=True
)

all_partition_logs = pd.concat(
    [heart_partition_logs, stroke_partition_logs],
    ignore_index=True
)

metrics_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Balanced Accuracy",
    "ROC-AUC",
    "Training Time (s)",
]

non_iid_summary_mean = (
    all_non_iid_results
    .groupby(["Dataset", "Method"])[metrics_columns]
    .mean()
)

non_iid_summary_std = (
    all_non_iid_results
    .groupby(["Dataset", "Method"])[metrics_columns]
    .std()
)

non_iid_mean_std_summary = non_iid_summary_mean.copy()

for metric in metrics_columns:
    non_iid_mean_std_summary[metric] = (
        non_iid_summary_mean[metric].map(lambda x: f"{x:.4f}")
        + " ± "
        + non_iid_summary_std[metric].map(lambda x: f"{x:.4f}")
    )

non_iid_mean_std_summary = non_iid_mean_std_summary.reset_index()

print("Non-IID mean ± standard deviation summary")
display(non_iid_mean_std_summary)

print("Non-IID hospital partition logs")
display(all_partition_logs.round(4))

Non-IID mean ± standard deviation summary


,Dataset,Method,Accuracy,Precision,Recall,F1-score,Balanced Accuracy,ROC-AUC,Training Time (s)
0,Heart Disease,Centralised LR,0.8400 ± 0.0480,0.8526 ± 0.0382,0.7929 ± 0.0814,0.8208 ± 0.0591,0.8371 ± 0.0500,0.8980 ± 0.0474,0.0018 ± 0.0004
1,Heart Disease,Federated LR (FedAvg),0.8467 ± 0.0506,0.8661 ± 0.0370,0.7929 ± 0.0924,0.8264 ± 0.0638,0.8433 ± 0.0530,0.8991 ± 0.0504,0.0900 ± 0.0032
2,Stroke Prediction,Centralised LR,0.7497 ± 0.0103,0.1250 ± 0.0058,0.8095 ± 0.0673,0.2165 ± 0.0107,0.7783 ± 0.0290,0.8445 ± 0.0225,0.0190 ± 0.0031
3,Stroke Prediction,Federated LR (FedAvg),0.7825 ± 0.0140,0.1320 ± 0.0102,0.7333 ± 0.0897,0.2236 ± 0.0178,0.7590 ± 0.0396,0.8368 ± 0.0302,0.1219 ± 0.0118


Non-IID hospital partition logs


,Dataset,Seed,Hospital,Samples,Negative Cases,Positive Cases,Positive Rate
0,Heart Disease,11,Hospital_A_high_positive,86,32,54,0.6279
1,Heart Disease,11,Hospital_B_low_positive,79,57,22,0.2785
2,Heart Disease,11,Hospital_C_mixed,72,39,33,0.4583
3,Heart Disease,22,Hospital_A_high_positive,86,32,54,0.6279
4,Heart Disease,22,Hospital_B_low_positive,79,57,22,0.2785
5,Heart Disease,22,Hospital_C_mixed,72,39,33,0.4583
6,Heart Disease,33,Hospital_A_high_positive,86,32,54,0.6279
7,Heart Disease,33,Hospital_B_low_positive,79,57,22,0.2785
8,Heart Disease,33,Hospital_C_mixed,72,39,33,0.4583
9,Heart Disease,44,Hospital_A_high_positive,86,32,54,0.6279


In [11]:
# Save non-IID experiment outputs

heart_non_iid_results.to_csv(
    "heart_non_iid_results.csv",
    index=False
)

stroke_non_iid_results.to_csv(
    "stroke_non_iid_results.csv",
    index=False
)

all_non_iid_results.to_csv(
    "all_non_iid_results.csv",
    index=False
)

non_iid_mean_std_summary.to_csv(
    "non_iid_mean_std_summary.csv",
    index=False
)

all_partition_logs.to_csv(
    "non_iid_hospital_partition_logs.csv",
    index=False
)

print("Saved files:")
print("- heart_non_iid_results.csv")
print("- stroke_non_iid_results.csv")
print("- all_non_iid_results.csv")
print("- non_iid_mean_std_summary.csv")
print("- non_iid_hospital_partition_logs.csv")

Saved files:
- heart_non_iid_results.csv
- stroke_non_iid_results.csv
- all_non_iid_results.csv
- non_iid_mean_std_summary.csv
- non_iid_hospital_partition_logs.csv
